# PromptSMILES Molecular Generation Tutorial

Generate molecules from fragment SMILES with the same GPT2 prior and PromptSMILES
samplers as `src/func/generation_promptsmiles_func.py`. Use the `promptsmiles` kernel.
The notebook performs inference; optional training is described in Section 7.
BRICS and RC_CMS fragmentation are supported for preparing interactive inputs.


## 0. Environment Setup

Run the following commands **in a terminal at the repository root** if the environment
is not already prepared. If it already exists, skip `conda create`.

```bash
conda create -n promptsmiles python=3.12 -y
conda run -n promptsmiles pip install -r requirements/promptsmiles_requirements.txt
conda run -n promptsmiles pip install -e .
conda run -n promptsmiles pip install ipykernel
conda run -n promptsmiles python -m ipykernel install --user --name promptsmiles --display-name "promptsmiles"
```

Select the **promptsmiles** kernel in Jupyter before running the cells below.
The first cell sets the working directory to the repository root so model, data,
and output paths work from this notebook's location in `tutorial/`.


In [ ]:
from pathlib import Path
import os

# Resolve paths consistently when opened from tutorial/ or the repository root.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'setup.py').is_file() and (path / 'src' / 'func').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')


## 1. Download Models

Use a PromptSMILES checkpoint trained by `src/train_model/train_promptsmiles.py`.
The download cell fetches the study's model archive; confirm that it contains the
configured `models/promptsmiles/gpt/{MODEL_VER}/{FRAG_METHOD}/best_model` directory.
If it does not, supply a local trained checkpoint using Section 7. The bare
`entropy/gpt2_zinc_87m` prior is not a replacement for a fine-tuned study checkpoint.
For a private repository, authenticate with Hugging Face before downloading.


In [ ]:
import os
import glob
import subprocess
from huggingface_hub import snapshot_download

# Download models from HuggingFace
snapshot_download(
    repo_id='sato-akinori/FFMG',
    allow_patterns='models/*',
    local_dir='.'
)

# Extract zip files
for zip_file in glob.glob('models/**/*.zip', recursive=True):
    subprocess.run(['unzip', '-o', zip_file, '-d', os.path.dirname(zip_file)], check=True)
    os.remove(zip_file)


print('Model download complete.')


## 2. Configuration

Configure the fragmentation method, generation parameters, etc.

In [ ]:
FRAG_METHOD = 'brics'       # 'brics' or 'rc_cms'; must match the checkpoint
MODEL_VER = 'finetuning'    # 'finetuning' or 'from_scratch'
GEN_METHOD = 'beam'         # 'beam' or 'sampling'; matches gen_promptsmiles.sh
N_SAMPLES = 50
NUM_BEAMS = 50
MAX_LENGTH = 256
RANDOM_SEED = 42

PROMPTSMILES_MODEL_PATH = f'models/promptsmiles/gpt/{MODEL_VER}/{FRAG_METHOD}/best_model'


### Relationship to the batch generation code

This notebook imports `GPT2PromptSampler`, `select_prompt_fragments`, `build_prompter`,
and `to_prediction_row` from `src/func/generation_promptsmiles_func.py`.
A single fragment uses scaffold decoration. Multiple fragments use linking only
when every fragment has exactly one attachment point; all requested fragments are
retained. Unsupported sets produce `INVALID_SMILES` in every candidate slot.

Defaults match `src/gen_mols/gen_promptsmiles.sh`: beam search, 50 samples, 50 beams,
maximum length 256, and seed 42. The Python CLI alone defaults to sampling.
Each row uses `RANDOM_SEED + row_index`, as in the batch implementation. To compare
sampling results, keep the same row order as well as the model and fragment inputs.
Custom fragments have no target molecule, so the dataset's `invalid_target` check
and target-based evaluation are not applicable. Generation errors are displayed and
marked `generation_error`, while row order and candidate slots are preserved.


## 3. Import Libraries and Helper Functions

In [ ]:
import sys
import os
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from transformers import AutoTokenizer, GPT2LMHeadModel, set_seed

from func.fragmentation import BRICSFragmentize, RandomFragmentize, PostProcessSelectFrags
from func.generation_promptsmiles_func import (
    GPT2PromptSampler, select_prompt_fragments, build_prompter, to_prediction_row,
)
from func.utility import INVALID_SMILES

def fragmentize_smiles(
    smiles: str, frag_method: str = 'brics', ratio: float = 0.6,
    big_ring_thres: int = 7, seed: int = 42,
) -> str | None:
    """Extract one processed fragment set using the dataset's fragmentation settings.

    Args:
        smiles: Input molecule as SMILES.
        frag_method: BRICS or random-cut fragmentation.
        ratio: Fraction of eligible bonds to cut for rc_cms.
        big_ring_thres: Ring-size threshold for rc_cms.
        seed: Random-cut seed.

    Returns:
        Dot-separated fragments, with multiplicities preserved, or None.
        This interactive example keeps all processed fragments; it does not
        sample subsets or apply the training-pair size filter in Smi2Sentences.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    if frag_method == 'brics':
        frags = BRICSFragmentize(mol, returnSmiles=False)
    elif frag_method == 'rc_cms':
        frags = RandomFragmentize(
            mol, returnSmiles=False, bigRingThres=big_ring_thres,
            rseed=seed, ratio=ratio, removeDummy=False,
        )
    else:
        raise ValueError(f'Unknown method: {frag_method}')
    if frags is None:
        return None
    pass_frags, _ = PostProcessSelectFrags(
        frags, smallCarbonFilter=True,
        trimRgroupOnRing=(frag_method == 'rc_cms'),
        uniquenize=False, returnAsSmi=True,
    )
    return pass_frags


def draw_mols(smiles_list, legends=None, mols_per_row=4, img_size=(300, 300)):
    """Draw molecules from a list of SMILES strings."""
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    mols = [m for m in mols if m is not None]
    if not mols:
        print('No valid molecules to draw.')
        return
    if legends is None:
        legends = [Chem.MolToSmiles(m) for m in mols]
    img = Draw.MolsToGridImage(
        mols[:12], molsPerRow=mols_per_row, subImgSize=img_size, legends=legends[:12]
    )
    display(img)


def load_promptsmiles_model(
    model_path: str, max_length: int, gen_method: str, num_beams: int,
) -> GPT2PromptSampler:
    """Load the prior and callbacks exactly as the repository generation script.

    Args:
        model_path: Checkpoint directory containing the model and tokenizer.
        max_length: Maximum total token sequence length.
        gen_method: Beam search or multinomial sampling.
        num_beams: Beam-search width.

    Returns:
        Sampler holding the model in evaluation mode and its tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = GPT2LMHeadModel.from_pretrained(model_path)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    print(f'Using device: {device}')
    return GPT2PromptSampler(model, tokenizer, device, max_length, gen_method, num_beams)


def generate_promptsmiles(
    fragment_sets: list[str], sampler: GPT2PromptSampler,
    n_samples: int, random_seed: int,
) -> pd.DataFrame:
    """Generate custom inputs with the repository's routing, seeds, and row policy.

    Args:
        fragment_sets: Ordered fragment SMILES sets, with multiplicities retained.
        sampler: Loaded prior with the selected decoding settings.
        n_samples: Candidate count per row.
        random_seed: Base seed; each input row adds its zero-based index.

    Returns:
        One row per input with sampler status, prompted fragments, and ranked
        prediction columns. Failed or unsupported rows keep INVALID_SMILES slots.
    """
    if n_samples < 1 or (sampler.gen_method == 'beam' and n_samples > sampler.num_beams):
        raise ValueError('Require positive n_samples, no greater than num_beams in beam mode.')
    set_seed(random_seed)
    records = []
    for index, fragment_set in enumerate(fragment_sets):
        status = 'unsupported'
        prompt_fragments = ''
        predictions = [INVALID_SMILES] * n_samples
        selected = select_prompt_fragments(fragment_set)
        if selected is not None:
            status, fragments = selected
            row_seed = random_seed + index
            torch.manual_seed(row_seed)
            try:
                prompter = build_prompter(status, fragments, sampler, n_samples, row_seed)
                sampled = prompter.sample()
            except Exception as error:
                status = 'generation_error'
                print(f'Row {index}: {type(error).__name__}: {error}')
            else:
                prompt_fragments = '.'.join(fragments)
                predictions = to_prediction_row(sampled, n_samples)
        records.append([fragment_set, status, prompt_fragments, *predictions])
    return pd.DataFrame(records, columns=[
        'fragment', 'sampler', 'prompt_fragments',
        *[f'prediction_{i + 1}' for i in range(n_samples)],
    ])


## 4. Input SMILES and Fragmentation

Enter an arbitrary SMILES and perform fragmentation.  
Modify `input_smiles` to try different molecules.

The cut ratio (0.6), ring threshold (7), small-carbon filtering, and ring trimming
match `src/gen_frags/rffmg_frags.py`; repeated fragments are retained. This example
uses one cut pattern and all processed fragments. Dataset construction additionally
samples subsets over multiple patterns and applies a molecule/fragment size filter.
Use the same stored fragment set when comparing with a dataset result.


In [ ]:
# ========================================
# Input SMILES (modifiable)
# ========================================
input_smiles = 'CC(C)Cc1ccc(C(C)C(=O)O)cc1'  # Ibuprofen

# Validate and canonicalize SMILES
mol = Chem.MolFromSmiles(input_smiles)
assert mol is not None, 'Invalid SMILES. Please enter a valid SMILES string.'
canonical_smi = Chem.MolToSmiles(mol)
print(f'Input molecule (canonical SMILES): {canonical_smi}')

# Fragmentation
pass_frags = fragmentize_smiles(canonical_smi, frag_method=FRAG_METHOD, seed=RANDOM_SEED)

if pass_frags is None:
    print('This molecule cannot be fragmented. Try a different SMILES or method.')
else:
    print(f'\n--- Fragmentation Results ---')
    print(f'Method: {FRAG_METHOD}')
    print(f'Fragments: {pass_frags}')
    print(f'Number of fragments: {len(pass_frags.split("."))}')

    frag_smiles = pass_frags.split('.')
    draw_mols(
        [canonical_smi] + frag_smiles,
        legends=['Input molecule'] + [f'Fragment {i+1}' for i in range(len(frag_smiles))]
    )

---
## 5. Molecule Generation from Fragmented Input

Generate molecules from the fragments obtained in Section 4.

In [ ]:
assert pass_frags is not None, 'Fragmentation failed. Please check Section 4.'

set_seed(RANDOM_SEED)
prompt_sampler = load_promptsmiles_model(
    PROMPTSMILES_MODEL_PATH, MAX_LENGTH, GEN_METHOD, NUM_BEAMS,
)
predictions = generate_promptsmiles([pass_frags], prompt_sampler, N_SAMPLES, RANDOM_SEED)
display(predictions)
draw_mols(predictions.filter(regex=r'^prediction_').iloc[0].tolist())


## 6. Generate Molecules from Custom Fragments

Use `*` for attachment points and `.` to separate fragments. A single fragment is
routed to scaffold decoration. Linking requires exactly one attachment point on
every fragment. The third example below is deliberately unsupported; it remains
in the output with `sampler="unsupported"` and invalid candidates.


In [ ]:
input_fragments = [
    '*c1ccccc1',             # Scaffold decoration
    '*c1ccccc1.*C(=O)O',    # Fragment linking
    '*CC*.*O',              # Unsupported: one fragment has two attachment points
]
custom_frags = []
for fragment_set in input_fragments:
    mols = [Chem.MolFromSmiles(fragment) for fragment in fragment_set.split('.')]
    if any(mol is None for mol in mols):
        raise ValueError(f'Invalid fragment SMILES: {fragment_set}')
    custom_frags.append('.'.join(Chem.MolToSmiles(mol) for mol in mols))
for fragment_set in custom_frags:
    draw_mols(fragment_set.split('.'))


### Generate from Custom Fragments


In [ ]:
set_seed(RANDOM_SEED)
prompt_sampler = load_promptsmiles_model(
    PROMPTSMILES_MODEL_PATH, MAX_LENGTH, GEN_METHOD, NUM_BEAMS,
)
gen_smiles_df = generate_promptsmiles(custom_frags, prompt_sampler, N_SAMPLES, RANDOM_SEED)
display(gen_smiles_df)
for _, row in gen_smiles_df.iterrows():
    print(f"{row['sampler']}: {row['fragment']}")
    draw_mols([row[f'prediction_{i + 1}'] for i in range(N_SAMPLES)])

output_dir = Path(f'results/promptsmiles/gpt/{MODEL_VER}/{FRAG_METHOD}/{GEN_METHOD}/custom_frag')
output_dir.mkdir(parents=True, exist_ok=True)
gen_smiles_df.to_csv(output_dir / 'gen_smiles.csv', index=False)


## 7. Optional Training with the Repository Scripts

The prior is trained on plain SMILES, without fragment prompts. Use the existing
script from a terminal at the repository root after preparing the datasets:

```bash
bash src/train_model/run_promptsmiles.sh
```

Set `FRAG_NAME` and `MODE` in the script to match this notebook. Notebook variables
do not change shell settings. `train_promptsmiles.py` reads the `smiles` column from
`data/promptsmiles/{FRAG_NAME}/normal`, randomizes each SMILES with a per-row seed,
and tokenizes `<bos> SMILES <eos>`. Sequences longer than 256 tokens are dropped.
It trains a causal language model using the script's parameters (50 epochs,
learning rate 1e-4, batch size 32, and warmup 10000). The Python entry point uses
seed 42 and early stopping patience 15 by default.

Fine-tuning initializes `entropy/gpt2_zinc_87m`; from-scratch mode initializes the
same config with random weights. Both use the prior's tokenizer. The best model
and tokenizer are saved to `models/promptsmiles/gpt/{MODE}/{FRAG_NAME}/best_model`.
Load that directory here for inference. No notebook cell starts training.
